In [1]:
import os
import shutil
import pickle
import boto3
from passwords import *

### Flask App Requirements
- ```requirements.txt``` (for installing correct versions of packages)
- ```cls_parser.pkl``` (for generating responses)
- ```app.py``` (for serving the Flask app)
- ```functions.py``` (helper functions for API)
- ```preprocessing.py``` (functions for shared preprocessing)
- ```api.py``` (for processing requests)

### Functions

In [2]:
# upload to s3
def download_from_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name, aws_session_token=None):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token,
    )
    # upload
    cls_client.download_file(
        str_project, 
        str_bucket_key, 
        str_local_path,
    )

### Constants

In [3]:
# project
try:
    str_project = os.getcwd().split('/')[4].replace('_','-')
except IndexError:
    str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
str_variant = 'noPTImodel10'

Project: 20231010-gen-xii


### Create ```app``` directory

In [4]:
str_dirname = 'app'
try:
    os.mkdir(str_dirname)
except FileExistsError:
    pass

### Place files appropriately

### Write ```requirements.txt``` to ```app/``` directory

In [5]:
%%writefile app/requirements.txt

waitress==2.1.1
flask==2.3.2

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
scikit_learn==0.24.1
boto3==1.24.59
catboost==1.0.4
xmltodict==0.14.2

Overwriting app/requirements.txt


### Write ```app.py``` to ```app/``` directory

In [6]:
%%writefile app/app.py

from flask import Flask, request, jsonify, Response
import pickle
import traceback
import logging
from waitress import serve
import pandas as pd 
pd.options.mode.chained_assignment = None # suppress warning

# set up logging
logging.basicConfig(
    filename='flask_app.log', 
    level=logging.DEBUG, 
    format='%(asctime)s %(levelname)s %(message)s',
)

# instantiate app
app = Flask(__name__)

# route the model to http://127.0.0.1:5000/
@app.route('/', methods=['GET','POST']) # GET for status code, POST for predictions
# logic for GET and POST requests
def predict():
    if request.method == 'GET':
        # log it
        logging.info('GET request received')
        # get status code
        int_status_code = Response(status=200).status_code
        # return
        return f'Status code: {int_status_code}'
    elif request.method == 'POST':
        try:
            # import parser
            str_message = 'Loading parser...'
            logging.info(str_message)
            print(str_message)
            print('')
            cls_parse_payload = pickle.load(open('cls_parser.pkl', 'rb'))
            
            # get payload
            str_message = 'Getting request...'
            logging.info(str_message)
            print(str_message)
            print('')
            dict_json_request = request.get_json()
            
            # parse payload
            str_message = 'Parsing payload...'
            logging.info(str_message)
            print(str_message)
            cls_parse_payload.get_data(dict_json_request)
            cls_parse_payload.engineer_pmt_hx()
            cls_parse_payload.shared_preprocessing()
            cls_parse_payload.generate_predictions()
            cls_parse_payload.apply_policies()
            cls_parse_payload.adverse_action()
            cls_parse_payload.counter_offers()
            cls_parse_payload.generate_output()
            
            # extract output
            str_message = 'Extracting output...'
            logging.info(str_message)
            print(str_message)
            print('')
            dict_output = cls_parse_payload.dict_output
            # return output_final
            return dict_output['output_final']
        except Exception as e:
            str_message = 'Exception occurred'
            logging.error(
                str_message, 
                exc_info=True,
            )
            #return traceback in json
            return jsonify({'error': str(e)})

# run app
if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000)

Overwriting app/app.py


### Copy ```cls_parser.pkl``` to```app/```

In [7]:
str_filename = 'cls_parser.pkl'
str_source = f'../../05_parser/01_single/output/{str_variant}/{str_filename}'
str_destination = f'./app/{str_filename}'
shutil.copyfile(str_source, str_destination)

'./app/cls_parser.pkl'

### Copy local code files

In [8]:
list_str_filenames = [
    'api.py',
    'functions.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
    'preprocessing.py',
]
for str_filename in list_str_filenames:
    str_source = f'../../05_parser/01_single/{str_filename}'
    str_destination = f'./app/{str_filename}'
    shutil.copyfile(str_source, str_destination)

### Copy code files in s3

In [9]:
# # preprocessing
# str_filename = 'preprocessing.py'
# str_local_path = f'./app/{str_filename}'
# str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
# download_from_s3(
#     aws_access_key_id=AWS_ACCESS_KEY_ID, 
#     aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
#     str_local_path=str_local_path, 
#     str_bucket_key=str_bucket_path, 
#     str_bucket_name=str_project, 
#     aws_session_token=None,
# )